# Syncing the viewer with your notebook

`vizarr.Viewer` reports what is in view, and can be moved from Python:

- **`viewer.viewport`**: a read-only dict describing the view of the first
  image: the well (and field) under the center of the view, the center and
  visible region *within that well* in pixels and physical units, the T/Z
  selection, and more. Updates as you pan, zoom or move the T/Z sliders.
- **`viewer.go_to(...)`**: move to a well, a position within it, and/or a zoom level.
- **`viewer.select(t=..., z=...)`**: change the timepoint / z-plane.

This notebook builds a small synthetic HCS plate with known "cells" (bright
spots), then wires the viewer to a live readout, navigation controls, and a
plot that follows the view. Run all cells, then scroll to the dashboard in
step 6.

In [ ]:
import pathlib

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import zarr
from IPython.display import display as show
from matplotlib.patches import Rectangle

import vizarr

## 1. Create a synthetic plate

A 3×4 plate (OME-NGFF 0.4) with `t`, `c`, `z`, `y`, `x` axes and 0.65 µm
pixels. Each well holds a random number of spots. Each spot is in focus at
one z-plane and grows brighter over time. Their true positions (in µm) are
kept in `spots` so we can plot them next to the image.

In [ ]:
PLATE = pathlib.Path("synthetic_plate.zarr")
ROWS, COLS = ["A", "B", "C"], ["1", "2", "3", "4"]
T, Z, SIZE, PIXEL_UM = 4, 5, 384, 0.65


def spot_brightness(spot: dict, t: int, z: int) -> float:
    """Brightness (0-1) of a spot at timepoint t and z-plane z."""
    growth = np.clip((t - spot["onset"] + 1) / T, 0, 1)
    focus = np.exp(-0.5 * (z - spot["z"]) ** 2)
    return float(growth * focus)


def random_spots(rng: np.random.Generator, well: str) -> list[dict]:
    """Spots in a well, with positions in µm."""
    return [
        {
            "well": well,
            "id": i,
            "x": rng.uniform(20, SIZE - 20) * PIXEL_UM,
            "y": rng.uniform(20, SIZE - 20) * PIXEL_UM,
            "z": int(rng.integers(0, Z)),
            "onset": int(rng.integers(0, T)),
            "radius": rng.uniform(4, 10),  # px
        }
        for i in range(int(rng.integers(3, 25)))
    ]


def render_well(rng: np.random.Generator, spots: list[dict]) -> np.ndarray:
    """Image of a well's spots, as a uint8 [t, c, z, y, x] array."""
    yy, xx = np.mgrid[0:SIZE, 0:SIZE]
    footprints = np.stack(
        [
            np.exp(
                -((xx - s["x"] / PIXEL_UM) ** 2 + (yy - s["y"] / PIXEL_UM) ** 2)
                / (2 * s["radius"] ** 2)
            )
            for s in spots
        ]
    )
    weights = np.array(
        [
            [[spot_brightness(s, t, z) for s in spots] for z in range(Z)]
            for t in range(T)
        ]
    )
    planes = 20 + 220 * np.einsum("tzs,syx->tzyx", weights, footprints)
    planes += rng.normal(0, 4, planes.shape)
    return np.clip(planes, 0, 255).astype(np.uint8)[:, np.newaxis]


def write_image(group: zarr.Group, data: np.ndarray) -> None:
    """Write an OME-NGFF 0.4 multiscale image with 3 resolution levels."""
    datasets = []
    for level in range(3):
        step = 2**level
        group.create_array(
            str(level), data=data[..., ::step, ::step], chunks=(1, 1, 1, 256, 256)
        )
        scale = [1, 1, 1, PIXEL_UM * step, PIXEL_UM * step]
        datasets.append(
            {
                "path": str(level),
                "coordinateTransformations": [{"type": "scale", "scale": scale}],
            }
        )
    axes = [
        {"name": "t", "type": "time"},
        {"name": "c", "type": "channel"},
        *({"name": n, "type": "space", "unit": "micrometer"} for n in "zyx"),
    ]
    group.attrs["multiscales"] = [
        {"version": "0.4", "axes": axes, "datasets": datasets}
    ]
    window = {"start": 0, "end": 180, "min": 0, "max": 255}
    group.attrs["omero"] = {
        "channels": [
            {"color": "00FF88", "label": "spots", "active": True, "window": window}
        ],
        "rdefs": {"model": "color", "defaultT": T - 1, "defaultZ": Z // 2},
    }


def make_plate(path: pathlib.Path, seed: int = 0) -> list[dict]:
    """Write the plate to `path` and return its spots."""
    rng = np.random.default_rng(seed)
    root = zarr.open_group(path, mode="w", zarr_format=2)
    root.attrs["plate"] = {
        "version": "0.4",
        "name": "synthetic plate",
        "rows": [{"name": r} for r in ROWS],
        "columns": [{"name": c} for c in COLS],
        "wells": [
            {"path": f"{r}/{c}", "rowIndex": i, "columnIndex": j}
            for i, r in enumerate(ROWS)
            for j, c in enumerate(COLS)
        ],
        "field_count": 1,
    }
    spots = []
    for r in ROWS:
        for c in COLS:
            well_spots = random_spots(rng, r + c)
            spots.extend(well_spots)
            well = root.require_group(f"{r}/{c}")
            well.attrs["well"] = {"version": "0.4", "images": [{"path": "0"}]}
            write_image(well.require_group("0"), render_well(rng, well_spots))
    return spots


spots = make_plate(PLATE)
f"{len(spots)} spots in {len(ROWS) * len(COLS)} wells"

## 2. Create the viewer

We display it in the dashboard in step 6. Display a viewer once only: each
displayed copy reports its own viewport.

In [ ]:
viewer = vizarr.Viewer(height="520px")
viewer.add_image(zarr.open_group(PLATE, mode="r"))

## 3. A live readout of `viewer.viewport`

`viewer.observe(callback, names="viewport")` calls `callback` whenever the view
changes (debounced to ~200 ms). `position` and `region` are relative to the
well under the center of the view. They are `None` when the center isn't
over a well, e.g. zoomed out in the gap between wells.

In [ ]:
readout = widgets.HTML()


def fmt(values: list[float] | None) -> str:
    """Format a point or box, or "-" if there is none."""
    return "-" if values is None else "(" + ", ".join(f"{v:.1f}" for v in values) + ")"


def update_readout(change: dict) -> None:
    """Show the main fields of the viewport."""
    vp = change["new"]
    position = vp["position"] or {}
    region = vp["region"] or {}
    rows = [
        ("well", vp["well"] or "-"),
        ("t / z", f"{vp['selection'].get('t')} / {vp['selection'].get('z')}"),
        ("center (px)", fmt(position.get("pixel"))),
        (f"center ({vp['unit']})", fmt(position.get("physical"))),
        ("visible region (µm)", fmt(region.get("physical"))),
        ("zoom", f"{vp['zoom']:.2f}"),
        ("wells in view", len(vp["visible_cells"])),
    ]
    cells = "".join(f"<tr><td><b>{k}</b></td><td>{v}</td></tr>" for k, v in rows)
    readout.value = f'<table style="font-family: monospace">{cells}</table>'


viewer.observe(update_readout, names="viewport")

## 4. Navigation controls

Buttons and sliders call `viewer.go_to(...)` and `viewer.select(...)`. The
T/Z sliders also follow the viewer's own sliders, via `viewport["selection"]`.

In [ ]:
well_picker = widgets.Dropdown(
    options=[r + c for r in ROWS for c in COLS],
    description="Well",
    layout={"width": "160px"},
)
spot_picker = widgets.Dropdown(description="Spot", layout={"width": "200px"})


def update_spot_options(*_: object) -> None:
    """List the spots of the chosen well."""
    spot_picker.options = [
        (f"#{s['id']} ({s['x']:.0f}, {s['y']:.0f}) µm", s["id"])
        for s in spots
        if s["well"] == well_picker.value
    ]
    spot_picker.value = spot_picker.options[0][1]


well_picker.observe(update_spot_options, names="value")
update_spot_options()


def go_to_well(_: widgets.Button) -> None:
    """Fit the chosen well into the view."""
    viewer.go_to(well_picker.value)


def go_to_spot(_: widgets.Button) -> None:
    """Center on the chosen spot, at the z-plane where it is in focus."""
    spot = next(
        s
        for s in spots
        if s["well"] == well_picker.value and s["id"] == spot_picker.value
    )
    # Physical coordinates within the well. Zoom 2 = 4 screen pixels per image pixel.
    viewer.go_to(spot["well"], x=spot["x"], y=spot["y"], units="physical", zoom=2)
    viewer.select(z=spot["z"])


def zoom_by(delta: float) -> None:
    """Zoom in (or out, if negative) by `delta` powers of two."""
    viewer.go_to(zoom=viewer.viewport.get("zoom", 0) + delta)


def pan_by(fx: float, fy: float) -> None:
    """Move by a fraction of the visible width/height, within the current well."""
    vp = viewer.viewport
    if not vp.get("position"):
        return  # the center isn't over a well
    x, y = vp["position"]["pixel"]
    x0, y0, x1, y1 = vp["region"]["pixel"]
    viewer.go_to(x=x + fx * (x1 - x0), y=y + fy * (y1 - y0))


def button(icon: str, action: object, description: str = "") -> widgets.Button:
    """Make a button that calls `action()` when clicked."""
    width = "auto" if description else "40px"
    b = widgets.Button(icon=icon, description=description, layout={"width": width})
    b.on_click(lambda _: action())
    return b


go_button = widgets.Button(description="Go to well", icon="crosshairs")
go_button.on_click(go_to_well)
spot_button = widgets.Button(description="Go to spot", icon="dot-circle-o")
spot_button.on_click(go_to_spot)
move_buttons = [
    button("minus", lambda: zoom_by(-1)),
    button("plus", lambda: zoom_by(1)),
    button("arrow-left", lambda: pan_by(-0.25, 0)),
    button("arrow-up", lambda: pan_by(0, -0.25)),
    button("arrow-down", lambda: pan_by(0, 0.25)),
    button("arrow-right", lambda: pan_by(0.25, 0)),
]

t_slider = widgets.IntSlider(description="t", max=T - 1, continuous_update=False)
z_slider = widgets.IntSlider(description="z", max=Z - 1, continuous_update=False)
t_slider.observe(lambda change: viewer.select(t=change["new"]), names="value")
z_slider.observe(lambda change: viewer.select(z=change["new"]), names="value")


def sync_sliders(change: dict) -> None:
    """Follow the viewer's T/Z selection."""
    selection = change["new"]["selection"]
    # Setting the same value is a no-op, so this doesn't loop back into select().
    t_slider.value = selection.get("t", 0)
    z_slider.value = selection.get("z", 0)


viewer.observe(sync_sliders, names="viewport")

controls = widgets.VBox(
    [
        widgets.HBox([well_picker, go_button, spot_picker, spot_button]),
        widgets.HBox([*move_buttons, t_slider, z_slider]),
    ]
)

## 5. A plot that follows the view

- **Top:** spots per well. The well under the center of the view is outlined
  and the counts of wells in view are highlighted.
- **Bottom:** the true spot positions in that well (µm), shaded by their
  brightness at the current t/z. The dashed box is the visible region and the
  cross is the center of the view.

In [ ]:
plot = widgets.Output(layout={"width": "440px"})
counts = np.zeros((len(ROWS), len(COLS)), dtype=int)
for s in spots:
    counts[ROWS.index(s["well"][0]), COLS.index(s["well"][1:])] += 1


def draw_plate(ax: plt.Axes, vp: dict) -> None:
    """Spot counts per well, highlighting the wells in view."""
    ax.imshow(counts, cmap="Greys", vmin=0, vmax=counts.max() * 1.6)
    ax.set_xticks(range(len(COLS)), COLS)
    ax.set_yticks(range(len(ROWS)), ROWS)
    ax.set_title("spots per well")
    for i, r in enumerate(ROWS):
        for j, c in enumerate(COLS):
            in_view = r + c in vp["visible_cells"]
            style = (
                {"color": "C2", "fontweight": "bold"} if in_view else {"color": "0.4"}
            )
            ax.text(j, i, counts[i, j], ha="center", va="center", **style)
    if vp["well"]:
        i, j = ROWS.index(vp["well"][0]), COLS.index(vp["well"][1:])
        outline = Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="C3", lw=2.5)
        ax.add_patch(outline)


def draw_well(ax: plt.Axes, vp: dict) -> None:
    """Spots in the well under the center of the view, and the visible region."""
    t, z = vp["selection"]["t"], vp["selection"]["z"]
    extent = SIZE * PIXEL_UM
    ax.set_xlim(0, extent)
    ax.set_ylim(extent, 0)  # image coordinates: y points down
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    if not vp["region"]:
        ax.set_title("center the view on a well")
        return
    well_spots = [s for s in spots if s["well"] == vp["well"]]
    ax.scatter(
        [s["x"] for s in well_spots],
        [s["y"] for s in well_spots],
        c=[spot_brightness(s, t, z) for s in well_spots],
        cmap="Greens",
        vmin=0,
        vmax=1,
        edgecolors="0.3",
        s=60,
    )
    x0, y0, x1, y1 = vp["region"]["physical"]
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ls="--", ec="C3"))
    ax.plot(*vp["position"]["physical"], "+", color="C3", markersize=14)
    ax.set_title(f"well {vp['well']} · t={t} z={z}")


def draw(change: dict) -> None:
    """Redraw the plot for the current viewport."""
    fig, (ax_plate, ax_well) = plt.subplots(
        2, 1, figsize=(4.4, 6.6), height_ratios=[1, 1.6]
    )
    draw_plate(ax_plate, change["new"])
    draw_well(ax_well, change["new"])
    fig.tight_layout()
    with plot:
        plot.clear_output(wait=True)
        show(fig)
    plt.close(fig)


viewer.observe(draw, names="viewport")

## 6. The dashboard

Pan and zoom in the viewer, or use the controls. The readout and the plot
should follow. Then try the code cells in step 7.

In [ ]:
viewer.layout.width = "620px"
widgets.VBox([controls, widgets.HBox([viewer, widgets.VBox([readout, plot])])])

## 7. Drive the viewer from code

Run these one at a time and watch the dashboard above.

In [ ]:
viewer.go_to("B2")  # fit well B2 into the view

In [ ]:
viewer.go_to("C4", x=50, y=60, units="physical", zoom=1)  # a point (µm) in well C4

In [ ]:
viewer.go_to(x=200, y=200)  # pixels, within the well under the center of the view

In [ ]:
viewer.select(t=0, z=4)  # earliest timepoint, top plane

In [ ]:
viewer.go_to(zoom=-2)  # zoom out to the whole plate (keeps the center)

In [ ]:
viewer.viewport  # the raw dict (as of the last update from the viewer)

## Notes

- `go_to`/`select` calls made before the viewer is displayed, or while the
  image is loading, are applied once it has loaded. "Run All" works.
- Well names ignore case and zero-padding (`"b2"`, `"B02"`). An unknown well
  is reported in the browser's developer console, not in Python.
- `viewport` describes the first image added to the viewer.
- The same works for remote plates, e.g.
  `viewer.add_image("https://.../plate.zarr")`. For a single well
  (`.../plate.zarr/B/2`), use `go_to(field=...)` to move between fields.